# RHINO Whip Antenna — FM & Frequency Axis Verification
**Author:** Jerry Mbulawa | **Supervisor:** Dr Phil Bull

## Purpose
Capture a wideband spectrum with the tunable desk whip antenna (no LNA) to:

1. Confirm FM broadcast band (87.5–108 MHz) is visible in the environment
2. Compare FM level against the EoR science band (60–85 MHz)
3. Provide a reference spectrum for comparison with the discone and subsequent verification tests

## Before running
- Connect the tunable whip antenna to **ADC_D** (no LNA, no attenuator)
- Set the whip length to approximately 75 cm (quarter-wave at 100 MHz)
- Run Cell 1 first, every session

---
| Cell | What it does |
|------|-------------|
| 1 | Setup — hardware init, DDR4 acquisition function |
| 2 | Signal health check |
| 3 | Wideband spectrum (0–500 MHz) |
| 4 | Zoom: 55–120 MHz — science band vs FM band |
| 5 | Save results |


In [1]:
# ================================================================
# CELL 1 — Setup
# Run this first every session. Initialises hardware and defines
# the DDR4 acquisition function (confirmed pattern from v6 Cell 6).
# ================================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os, datetime
from pathlib import Path

# ── Output directory ──────────────────────────────────────────────
OUT = Path('whip_verification')
OUT.mkdir(exist_ok=True)
print(f'Output directory: {OUT.resolve()}')

# ── Hardware constants ────────────────────────────────────────────
FS_MHZ           = 4423.680      # ADC sample rate (MHz) — hard-coded, do not use soc.readouts[ch].fs
N_FFT            = 16384         # coarse FFT size
N_TAPS           = 4             # PFB taps (not used here, for reference)
SAMPLES_PER_XFER = 256           # samples per DDR4 transfer
NT_COARSE        = 68            # transfers per coarse spectrum (N_FFT/256 + headroom)
ADC_CH           = 0             # ADC channel (ADC_D)
ADC_FULL_SCALE   = 32767         # 15-bit ADC
CLIP_THRESHOLD   = 0.95          # fraction of full scale = clipping

# ── Derived frequency axes ────────────────────────────────────────
DF_MHZ    = FS_MHZ / N_FFT      # ≈ 0.270 MHz/bin
NYQUIST   = FS_MHZ / 2          # ≈ 2211.8 MHz
freq      = np.fft.rfftfreq(N_FFT) * FS_MHZ   # MHz, shape (8193,)

# ── Band markers ─────────────────────────────────────────────────
SCI_LO, SCI_HI = 60.0,  85.0    # EoR science band (MHz)
FM_LO,  FM_HI  = 87.5, 108.0    # FM broadcast band (MHz)

# ── QICK initialisation ───────────────────────────────────────────
HARDWARE_CONNECTED = False
try:
    from qick import *
    from qick.averager_program import AveragerProgram

    soc = QickSoc()

    # DDR4TriggerProgram — ddr4=True is mandatory.
    # Without ddr4=True, soc.ddr4_buf returns stale zeros on every call.
    class DDR4TriggerProgram(AveragerProgram):
        def initialize(self):
            self.declare_readout(ch=ADC_CH, length=1000, freq=0, gen_ch=None)
            self.synci(200)
        def body(self):
            self.trigger(adcs=[ADC_CH], ddr4=True, adc_trig_offset=100)
            self.wait_all()
            self.sync_all(self.us2cycles(1.0))

    _pcfg = {'ro_ch': ADC_CH, 'readout_length': 1000,
             'adc_trig_offset': 100, 'soft_avgs': 1,
             'reps': 1, 'relax_delay': 1.0}
    prog = DDR4TriggerProgram(soc, _pcfg)

    HARDWARE_CONNECTED = True
    print(f'QICK connected — FS = {FS_MHZ:.3f} MHz')
except Exception as e:
    print(f'QICK not connected ({e}) — offline mode')

# ── DDR4 capture ──────────────────────────────────────────────────
# soc.readouts[ch].reset_buf()       — DOES NOT EXIST in QICK 0.2.388
# soc.readouts[ch].transfer_avg_buf() — DOES NOT EXIST in QICK 0.2.388
# Correct path: soc.ddr4_buf with set_switch / arm / get_mem
def _ddr4_capture_raw(nt=NT_COARSE):
    """Single DDR4 capture — exact pattern from v6 Cell 6."""
    _cfg = soc.get_cfg()
    soc.ddr4_buf.set_switch(_cfg['readouts'][ADC_CH]['avgbuf_fullpath'])
    soc.clear_ddr4()
    soc.ddr4_buf.arm(nt=nt)
    prog.acquire(soc, load_pulses=False, progress=False)
    raw = soc.ddr4_buf.get_mem(nt=nt)
    return (raw[:, 0] if raw.ndim == 2 else raw).astype(np.float32)

# ── Spectrum functions ────────────────────────────────────────────
def _hann_spectrum(samples):
    """Single Hann-windowed FFT spectrum, returns power in dB."""
    win  = np.hanning(N_FFT).astype(np.float64)
    enbw = 1.5
    S    = np.fft.rfft(samples[:N_FFT].astype(np.float64) * win)
    P    = np.abs(S) ** 2 / float(np.sum(win ** 2))
    return (10 * np.log10(np.maximum(P, 1e-30)) + 10 * np.log10(enbw)).astype(np.float32)

def acquire_spectrum(n_frames=300, label=''):
    """
    Average n_frames coarse Hann-windowed spectra.
    Returns (freq_mhz, power_db).
    """
    if not HARDWARE_CONNECTED:
        raise RuntimeError('Hardware not connected')
    acc = np.zeros(N_FFT // 2 + 1, dtype=np.float64)
    for k in range(n_frames):
        raw  = _ddr4_capture_raw(NT_COARSE)
        acc += _hann_spectrum(raw)
        if (k + 1) % 50 == 0:
            print(f'  {k+1}/{n_frames} frames...', end='\r')
    print(f'  {n_frames}/{n_frames} frames — done        ')
    spec_db = (acc / n_frames).astype(np.float32)
    print(f'  Acquired {n_frames} frames — {label}')
    return freq.copy(), spec_db

print()
print(f'Channel width : {DF_MHZ*1000:.1f} kHz/bin')
print(f'Science band  : {SCI_LO:.0f}–{SCI_HI:.0f} MHz')
print(f'FM band       : {FM_LO:.1f}–{FM_HI:.0f} MHz')
print(f'Hardware      : {"CONNECTED" if HARDWARE_CONNECTED else "OFFLINE"}')
print()
print('Setup complete — ready to run Cell 2.')


Output directory: /home/xilinx/jupyter_notebooks/whip_verification


QICK connected — FS = 4423.680 MHz

Channel width : 270.0 kHz/bin
Science band  : 60–85 MHz
FM band       : 87.5–108 MHz
Hardware      : CONNECTED

Setup complete — ready to run Cell 2.


## Cell 2 — Signal health check
Run immediately after connecting the whip.
Checks RMS level, clipping, and that data is live.

**Expected (desk whip, no LNA):** RMS 100–3000 ADU, 0% clipping.

In [2]:
# ================================================================
# CELL 2 — Signal health check
# ================================================================
if not HARDWARE_CONNECTED:
    print('SKIP — hardware not connected')
else:
    print('Health check | whip antenna | no LNA')
    print('='*45)

    buffers = []
    for k in range(3):
        raw  = _ddr4_capture_raw(NT_COARSE)
        rms  = float(np.sqrt(np.mean(raw ** 2)))
        clip = float(np.mean(np.abs(raw) > CLIP_THRESHOLD * ADC_FULL_SCALE)) * 100
        buffers.append(raw[:500].copy())
        print(f'  Capture {k+1}: RMS = {rms:.1f} ADU   clip = {clip:.2f}%')

    is_live  = not np.allclose(buffers[0], buffers[1])
    rms_f    = float(np.sqrt(np.mean(buffers[0] ** 2)))
    clip_f   = float(np.mean(np.abs(buffers[0]) > CLIP_THRESHOLD * ADC_FULL_SCALE)) * 100

    print(f'\n  Live data   : {is_live}')
    print(f'  Final RMS   : {rms_f:.1f} ADU')
    print(f'  Clipping    : {clip_f:.2f}%')

    if not is_live:
        print('\nFAIL — DDR4 returning stale data.')
        print('Fix : restart kernel and re-run from Cell 1.')
    elif clip_f > 5.0:
        print('\nWARN — high clipping, check antenna and connections.')
    elif rms_f < 0.5:
        print('\nWARN — very low signal, check whip is connected to ADC_D.')
    else:
        print('\nPASS — signal healthy, proceed to Cell 3.')


Health check | whip antenna | no LNA
  Capture 1: RMS = 8.5 ADU   clip = 0.00%
  Capture 2: RMS = 13.1 ADU   clip = 0.00%
  Capture 3: RMS = 8.5 ADU   clip = 0.00%

  Live data   : True
  Final RMS   : 7.9 ADU
  Clipping    : 0.00%

PASS — signal healthy, proceed to Cell 3.


## Cell 3 — Wideband spectrum (0–500 MHz)
Captures a full wideband spectrum and plots it with the science band and FM band marked.

**What to look for:**
- FM band (87.5–108 MHz) should appear as a bright elevated region
- Science band (60–85 MHz) should be visible just below
- If FM is visible here, it is real environmental FM, not a board artefact

In [3]:
# ================================================================
# CELL 3 — Wideband spectrum (0–500 MHz)
# ================================================================
N_FRAMES_WB = 300   # 300 averaged frames — adjust if slow

if HARDWARE_CONNECTED:
    print(f'Acquiring {N_FRAMES_WB} frames for wideband spectrum...')
    freq_wb, spec_wb = acquire_spectrum(N_FRAMES_WB, label='whip wideband')
    np.savez(OUT / 'whip_wideband_spectrum.npz', freq=freq_wb, spec=spec_wb)
else:
    load_p = OUT / 'whip_wideband_spectrum.npz'
    if load_p.exists():
        d = np.load(load_p)
        freq_wb, spec_wb = d['freq'], d['spec']
        print(f'Loaded offline: {load_p}')
    else:
        print('No saved data found. Run with hardware connected.')
        freq_wb = freq.copy()
        spec_wb = np.random.randn(len(freq)) * 3 + 60  # placeholder
        print('Using placeholder data for structure check only.')

# ── Summary statistics ────────────────────────────────────────────
plot_mask = freq_wb <= 500
sci_mask  = (freq_wb >= SCI_LO) & (freq_wb <= SCI_HI)
fm_mask   = (freq_wb >= FM_LO)  & (freq_wb <= FM_HI)
noise_mask= (freq_wb >= 10)     & (freq_wb <= 50)     # quiet reference band

noise_floor = float(np.median(spec_wb[noise_mask]))
sci_level   = float(np.mean(spec_wb[sci_mask]))
fm_level    = float(np.mean(spec_wb[fm_mask]))
fm_vs_sci   = fm_level - sci_level

print(f'\nSpectral levels:')
print(f'  Noise floor (10–50 MHz reference) : {noise_floor:.1f} dB')
print(f'  Science band mean (60–85 MHz)      : {sci_level:.1f} dB')
print(f'  FM band mean (87.5–108 MHz)        : {fm_level:.1f} dB')
print(f'  FM above science band              : {fm_vs_sci:+.1f} dB')

if fm_vs_sci > 3:
    print(f'\n  FM is {fm_vs_sci:.1f} dB above the science band — FM pickup confirmed.')
elif fm_vs_sci > 0:
    print(f'\n  FM is {fm_vs_sci:.1f} dB above the science band — weak FM pickup.')
else:
    print(f'\n  FM band not elevated above science band — FM not clearly visible.')

# ── Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(freq_wb[plot_mask], spec_wb[plot_mask],
        color='#1f77b4', lw=0.8, label='Whip antenna (no LNA)')
ax.axvspan(SCI_LO, SCI_HI, alpha=0.15, color='gold',
           label=f'Science band {SCI_LO:.0f}–{SCI_HI:.0f} MHz')
ax.axvspan(FM_LO, FM_HI, alpha=0.12, color='salmon',
           label=f'FM band {FM_LO:.1f}–{FM_HI:.0f} MHz')
ax.axhline(noise_floor, color='grey', lw=0.7, ls='--',
           label=f'Noise floor ({noise_floor:.1f} dB)')
ax.set_xlim(0, 500)
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Power (dB)')
ax.set_title(
    f'Wideband Spectrum — Desk Whip Antenna (no LNA)\n'
    f'FM: {fm_level:.1f} dB   Science band: {sci_level:.1f} dB   '
    f'FM above science: {fm_vs_sci:+.1f} dB',
    fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(ticker.MultipleLocator(50))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
fig.tight_layout()
fig.savefig(OUT / f'whip_wideband_{ts}.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nFigure saved: {OUT}/whip_wideband_{ts}.png')
print('PASS — wideband spectrum complete.')


Acquiring 300 frames for wideband spectrum...
  300/300 frames — done        
  Acquired 300 frames — whip wideband

Spectral levels:
  Noise floor (10–50 MHz reference) : 20.2 dB
  Science band mean (60–85 MHz)      : 19.5 dB
  FM band mean (87.5–108 MHz)        : 18.6 dB
  FM above science band              : -0.9 dB

  FM band not elevated above science band — FM not clearly visible.

Figure saved: whip_verification/whip_wideband_20260520_081926.png
PASS — wideband spectrum complete.


## Cell 4 — Zoom: 55–120 MHz
Shows the science band and FM band side by side at full resolution.

**What to look for:**
- Individual FM stations visible as narrow peaks at known frequencies
  (e.g. local stations at 96.2, 100.1, 103.0 MHz and so on)
- Science band (60–85 MHz) level relative to FM
- Frequency axis accuracy: FM peaks should sit at their known broadcast frequencies

In [4]:
# ================================================================
# CELL 4 — Zoom: 55–120 MHz — science band vs FM band
# ================================================================
if 'spec_wb' not in dir() or spec_wb is None:
    print('SKIP — run Cell 3 first.')
else:
    zoom_mask = (freq_wb >= 55) & (freq_wb <= 120)
    fz = freq_wb[zoom_mask]
    sz = spec_wb[zoom_mask]

    # ── Find FM peaks in the FM band ─────────────────────────────
    fm_band_mask = (fz >= FM_LO) & (fz <= FM_HI)
    if fm_band_mask.any():
        fm_section = sz[fm_band_mask]
        fm_freqs   = fz[fm_band_mask]
        fm_median  = float(np.median(fm_section))

        # Find peaks > 3 dB above FM median
        is_peak = np.zeros(len(fm_section), dtype=bool)
        for i in range(1, len(fm_section) - 1):
            if (fm_section[i] > fm_section[i-1] and
                fm_section[i] > fm_section[i+1] and
                fm_section[i] > fm_median + 3.0):
                is_peak[i] = True

        # Suppress peaks within 0.5 MHz of a stronger one
        peak_idxs = np.where(is_peak)[0]
        keep = []
        for idx in peak_idxs:
            if not keep or (fm_freqs[idx] - fm_freqs[keep[-1]]) > 0.5:
                keep.append(idx)
            elif fm_section[idx] > fm_section[keep[-1]]:
                keep[-1] = idx
        peak_idxs = keep

        print('FM station peaks detected:')
        for idx in peak_idxs:
            print(f'  {fm_freqs[idx]:.2f} MHz   {fm_section[idx]:.1f} dB   '
                  f'({fm_section[idx] - fm_median:+.1f} dB above FM median)')
        print(f'  Total: {len(peak_idxs)} stations visible')
    else:
        print('FM band not in zoom range.')
        peak_idxs = []

    # ── Plot ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: full 55–120 MHz view
    ax = axes[0]
    ax.plot(fz, sz, color='#1f77b4', lw=0.9)
    ax.axvspan(SCI_LO, SCI_HI, alpha=0.15, color='gold',
               label=f'Science band {SCI_LO:.0f}–{SCI_HI:.0f} MHz')
    ax.axvspan(FM_LO, FM_HI, alpha=0.12, color='salmon',
               label=f'FM band {FM_LO:.1f}–{FM_HI:.0f} MHz')
    # Mark detected FM peaks
    if fm_band_mask.any() and peak_idxs:
        for idx in peak_idxs:
            ax.axvline(fm_freqs[idx], color='red', lw=0.6, alpha=0.6)
            ax.annotate(f'{fm_freqs[idx]:.1f}',
                        xy=(fm_freqs[idx], fm_section[idx]),
                        xytext=(2, 4), textcoords='offset points',
                        fontsize=6, color='red', rotation=90)
    ax.set_xlim(55, 120)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB)')
    ax.set_title('Science band vs FM band | 55–120 MHz', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    # Right: science band only
    ax = axes[1]
    sci_mask2 = (freq_wb >= SCI_LO - 2) & (freq_wb <= SCI_HI + 2)
    ax.plot(freq_wb[sci_mask2], spec_wb[sci_mask2],
            color='darkorange', lw=0.9, label='Whip antenna (no LNA)')
    ax.axvspan(SCI_LO, SCI_HI, alpha=0.12, color='gold',
               label=f'Science band {SCI_LO:.0f}–{SCI_HI:.0f} MHz')
    ax.axhline(float(np.mean(spec_wb[sci_mask2])),
               color='grey', lw=0.7, ls='--',
               label=f'Mean: {float(np.mean(spec_wb[sci_mask2])):.1f} dB')
    ax.set_xlim(SCI_LO - 2, SCI_HI + 2)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB)')
    ax.set_title(f'EoR Science band zoom | {SCI_LO:.0f}–{SCI_HI:.0f} MHz', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

    ts2 = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    fig.suptitle(
        f'Whip Antenna — Science Band vs FM Band\n'
        f'Science: {sci_level:.1f} dB   FM: {fm_level:.1f} dB   '
        f'FM above science: {fm_vs_sci:+.1f} dB',
        fontsize=10)
    fig.tight_layout()
    fig.savefig(OUT / f'whip_zoom_{ts2}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figure saved: {OUT}/whip_zoom_{ts2}.png')
    print('PASS — zoom complete.')


FM station peaks detected:
  Total: 0 stations visible
Figure saved: whip_verification/whip_zoom_20260520_081939.png
PASS — zoom complete.


## Cell 5 — Save and summary
Saves raw spectrum arrays and prints a one-line summary.

In [5]:
# ================================================================
# CELL 5 — Save results and print summary
# ================================================================
import json as _json

summary = {}

if 'spec_wb' in dir() and spec_wb is not None:
    ts3  = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    np.savez(OUT / f'whip_spectrum_{ts3}.npz', freq=freq_wb, spec=spec_wb)
    print(f'Spectrum saved: {OUT}/whip_spectrum_{ts3}.npz')

    summary = {
        'timestamp'            : ts3,
        'antenna'              : 'desk whip (no LNA)',
        'n_frames'             : int(N_FRAMES_WB),
        'noise_floor_db'       : round(float(noise_floor), 2),
        'science_band_mean_db' : round(float(sci_level), 2),
        'fm_band_mean_db'      : round(float(fm_level), 2),
        'fm_above_science_db'  : round(float(fm_vs_sci), 2),
        'fm_peaks_detected'    : (len(peak_idxs) if 'peak_idxs' in dir() else 'N/A'),
    }

    with open(OUT / f'whip_summary_{ts3}.json', 'w') as _f:
        _json.dump(summary, _f, indent=2)
    print(f'Summary saved: {OUT}/whip_summary_{ts3}.json')
else:
    print('No spectrum data to save — run Cells 3–4 first.')

print()
print('='*55)
print('  WHIP ANTENNA VERIFICATION — SUMMARY')
print('='*55)
for k, v in summary.items():
    print(f'  {k:<30} : {v}')
print('='*55)

if summary:
    if summary['fm_above_science_db'] > 3:
        print('\n  RESULT: FM pickup confirmed in environment.')
        print('  FM is {:.1f} dB above the science band.'.format(summary['fm_above_science_db']))
        print('  Proceed to Test 1 (50Ω terminator) to determine')
        print('  whether this reaches the ADC via cable or direct pickup.')
    else:
        print('\n  RESULT: FM band not significantly elevated.')
        print('  This may indicate poor antenna coupling above 85 MHz.')
        print('  Try extending the whip element to ~1.4 m (quarter-wave at 54 MHz)')
        print('  or check the SMA connection.')


Spectrum saved: whip_verification/whip_spectrum_20260520_081948.npz
Summary saved: whip_verification/whip_summary_20260520_081948.json

  WHIP ANTENNA VERIFICATION — SUMMARY
  timestamp                      : 20260520_081948
  antenna                        : desk whip (no LNA)
  n_frames                       : 300
  noise_floor_db                 : 20.21
  science_band_mean_db           : 19.49
  fm_band_mean_db                : 18.58
  fm_above_science_db            : -0.92
  fm_peaks_detected              : 0

  RESULT: FM band not significantly elevated.
  This may indicate poor antenna coupling above 85 MHz.
  Try extending the whip element to ~1.4 m (quarter-wave at 54 MHz)
  or check the SMA connection.
